# IMPECT data · one-match walkthrough

**Optional learning notebook.** Inspect the source structures on one match before reading the main season analysis. Its variables are not required by any other notebook.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [1]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)



Project: /workspace/scratch/491a21a4af19/organized_project/football-no8-recruitment


## Inspect IMPECT iterations

Read the provider season catalogue. The final analysis pins iteration 743 explicitly instead of choosing the first catalogue row.

In [2]:
iterations_data = metadata("iterations.json")
print(json.dumps(iterations_data, indent=2, ensure_ascii=False)[:4000])


[
  {
    "id": 743,
    "season": "23/24",
    "competition": {
      "name": "1. Bundesliga",
      "id": 2,
      "type": "League",
      "countryId": 466,
      "gender": "MALE"
    },
    "dataVersion": "V3",
    "lastChangeTimestamp": "2025-10-06T08:38:54.475988Z",
    "idMappings": [
      {
        "heim_spiel": [
          "51884"
        ]
      },
      {
        "skill_corner": [
          "551"
        ]
      }
    ]
  }
]


## Select Bundesliga 2023/24

Load match metadata for iteration 743. Pinning the identifier protects the analysis from catalogue reordering.

In [3]:
iteration_id = ITERATION_ID
impect_matches_data = metadata(f"matches/matches_{iteration_id}.json")
print("Iteration:", iteration_id, "Matches:", len(impect_matches_data))
display(pd.json_normalize(impect_matches_data).head())


Iteration: 743 Matches: 306
   iterationId      id  ...  matchDay.index  matchDay.name
0          743  122838  ...               0    1. Spieltag
1          743  122839  ...               0    1. Spieltag
2          743  122840  ...               0    1. Spieltag
3          743  122841  ...               0    1. Spieltag
4          743  122842  ...               0    1. Spieltag

[5 rows x 10 columns]


## Check the season scope

Verify unique available matches and count each team’s appearances. The expected season has 306 matches and 18 teams.

In [4]:
impect_matches = pd.json_normalize(impect_matches_data)

print("Match records:", len(impect_matches))
print("Unique match IDs:", impect_matches["id"].nunique())
print("Missing match IDs:", impect_matches["id"].isna().sum())

print("\nAvailability:")
print(impect_matches["available"].value_counts(dropna=False))

# Count each team's appearances in available, unique matches.
available_matches = (
    impect_matches.loc[impect_matches["available"].eq(True)]
    .drop_duplicates(subset="id")
)

team_appearances = pd.concat(
    [
        available_matches["homeSquadId"],
        available_matches["awaySquadId"],
    ],
    ignore_index=True,
)

team_coverage = (
    team_appearances.value_counts()
    .rename_axis("squad_id")
    .reset_index(name="available_matches")
    .sort_values(["available_matches", "squad_id"])
    .reset_index(drop=True)
)

print("\nTeams represented:", len(team_coverage))
display(team_coverage)

Match records: 306
Unique match IDs: 306
Missing match IDs: 0

Availability:
available
True    306
Name: count, dtype: int64

Teams represented: 18
    squad_id  available_matches
0         27                 34
1         29                 34
2         30                 34
3         31                 34
4         32                 34
5         33                 34
6         34                 34
7         35                 34
8         36                 34
9         37                 34
10        38                 34
11        39                 34
12        41                 34
13        42                 34
14        46                 34
15       416                 34
16       432                 34
17       446                 34


## Read the team directory

Inspect squad IDs and names before using them as a lookup.

In [5]:
squads_data = metadata(f"squads/squads_{iteration_id}.json")
print("Squads:", len(squads_data))


Squads: 18


## Inspect one match’s events

Use the earliest available match as a small example before processing a full season.

In [6]:
# Select the earliest available match.
example_match = (
    available_matches
    .sort_values(["scheduledDate", "id"])
    .iloc[0]
)

example_match_id = int(example_match["id"])

events_url = (
    f"{IMPECT_BASE_URL}/events/events_{example_match_id}.json"
)

impect_events_data, cache_status = load_or_download_json(
    events_url, RAW_DIR / "events" / f"events_{example_match_id}.json")

print("Match ID:", example_match_id)
print("Python type:", type(impect_events_data).__name__)

# Display a small sample without assuming the JSON structure.
sample = (
    impect_events_data[:2]
    if isinstance(impect_events_data, list)
    else impect_events_data
)

print(json.dumps(sample, indent=2, ensure_ascii=False)[:6000])

Match ID: 122838
Python type: list
[
  {
    "index": 0,
    "id": 4858179098,
    "gameTime": {
      "gameTime": "00:00.0000",
      "gameTimeInSec": 0.0
    },
    "squadId": 33,
    "player": {
      "id": 204,
      "position": "RIGHT_WINGER",
      "positionSide": "RIGHT"
    },
    "pressure": 0.0,
    "actionType": "KICK_OFF",
    "start": {
      "coordinates": {
        "x": 0.0,
        "y": 0.0
      },
      "adjCoordinates": {
        "x": 0.0,
        "y": 0.0
      },
      "packingZone": "CBC",
      "pitchPosition": "MIDDLE_THIRD",
      "lane": "CENTER"
    },
    "end": null,
    "action": "KICKOFF_WHISTLE",
    "phase": "SET_PIECE",
    "duel": null,
    "shot": null,
    "pass": {
      "distance": 0.0,
      "angle": null,
      "receiver": null
    },
    "currentAttackingSquadId": 33,
    "bodyPart": "FOOT",
    "bodyPartExtended": "FOOT",
    "previousPassHeight": null,
    "duration": 0.332,
    "opponents": 11,
    "pxT": {
      "team": 0.005388,
      "opp

## Flatten nested event JSON

Turn nested fields into columns such as player.id and start.adjCoordinates.x.

In [7]:
impect_events = pd.json_normalize(impect_events_data)

print("Events:", len(impect_events))
print("Columns:", len(impect_events.columns))

columns_to_view = [
    "index",
    "id",
    "gameTime.gameTimeInSec",
    "squadId",
    "player.id",
    "player.position",
    "actionType",
]

display(impect_events[columns_to_view].head(15))

Events: 2978
Columns: 68
    index          id  ...          player.position         actionType
0       0  4858179098  ...             RIGHT_WINGER           KICK_OFF
1       1  4858179099  ...                      NaN           NO_VIDEO
2       2  4858179100  ...               GOALKEEPER          RECEPTION
3       3  4858179101  ...               GOALKEEPER            DRIBBLE
4       4  4858179102  ...               GOALKEEPER               PASS
5       5  4858179103  ...  RIGHT_WINGBACK_DEFENDER  LOOSE_BALL_REGAIN
6       6  4858179104  ...  RIGHT_WINGBACK_DEFENDER               PASS
7       7  4858179105  ...       ATTACKING_MIDFIELD          RECEPTION
8       8  4858179106  ...       ATTACKING_MIDFIELD          CLEARANCE
9       9  4858179107  ...         CENTRAL_DEFENDER  LOOSE_BALL_REGAIN
10     10  4858179108  ...         CENTRAL_DEFENDER               PASS
11     11  4858179109  ...         CENTRAL_DEFENDER                OUT
12     12  4858179110  ...  RIGHT_WINGBACK_DEFENDER 

## Inspect event and position labels

Count the provider’s recorded action and position categories so later filters use actual source values.

In [8]:
print("ACTION TYPES")
display(
    impect_events["actionType"]
    .value_counts(dropna=False)
    .rename_axis("action_type")
    .reset_index(name="event_count")
)

print("RECORDED POSITIONS")
display(
    impect_events["player.position"]
    .value_counts(dropna=False)
    .rename_axis("position")
    .reset_index(name="event_count")
)

ACTION TYPES
          action_type  event_count
0                PASS          918
1           RECEPTION          831
2             DRIBBLE          735
3   LOOSE_BALL_REGAIN          111
4                 OUT           52
5        INTERCEPTION           51
6           CLEARANCE           38
7         GROUND_DUEL           37
8               BLOCK           31
9                SHOT           30
10          FREE_KICK           29
11               FOUL           27
12           THROW_IN           25
13          GOAL_KICK           17
14           GK_CATCH           12
15           NO_VIDEO           10
16             CORNER            6
17           KICK_OFF            5
18            OFFSIDE            4
19               GOAL            4
20            GK_SAVE            3
21      FINAL_WHISTLE            2
RECORDED POSITIONS
                  position  event_count
0         CENTRAL_DEFENDER          755
1         DEFENSE_MIDFIELD          616
2   LEFT_WINGBACK_DEFENDER          305
3  

## Explore midfield actions in one match

Summarise actions by team, player and recorded midfield position. This sample does not select the final season candidate pool.

In [9]:
# These are candidate position labels, not our final No. 8 definition.
midfield_positions = [
    "DEFENSE_MIDFIELD",
    "ATTACKING_MIDFIELD",
]

midfield_events = impect_events.loc[
    impect_events["player.position"].isin(midfield_positions)
].copy()

midfield_summary = (
    midfield_events.groupby(
        ["squadId", "player.id", "player.position", "actionType"]
    )
    .size()
    .unstack("actionType", fill_value=0)
)

# Display a few action types relevant to our investigation.
actions_to_view = [
    "PASS",
    "RECEPTION",
    "DRIBBLE",
    "INTERCEPTION",
    "LOOSE_BALL_REGAIN",
    "GROUND_DUEL",
    "SHOT",
]

display(
    midfield_summary.reindex(
        columns=actions_to_view,
        fill_value=0,
    ).reset_index()
)

actionType  squadId  player.id  ... GROUND_DUEL  SHOT
0              33.0       98.0  ...           0     3
1              33.0      216.0  ...           0     0
2              33.0     1053.0  ...           1     3
3              33.0     7594.0  ...           2     0
4              33.0    53703.0  ...           2     3
5              38.0     1099.0  ...           3     1
6              38.0     3153.0  ...           1     1
7              38.0     4366.0  ...           0     0
8              38.0     6469.0  ...           2     0
9              38.0    13599.0  ...           3     1
10             38.0    23157.0  ...           3     0

[11 rows x 10 columns]


## List column names

Show the available fields without printing the whole event table.

In [10]:
print("\n".join(impect_events.columns))

index
id
squadId
pressure
actionType
end
action
phase
duel
shot
currentAttackingSquadId
bodyPart
bodyPartExtended
previousPassHeight
duration
opponents
distanceToGoal
result
periodId
pressingPlayerId
fouledPlayerId
distanceToOpponent
setPiece
sequenceIndex
inferredSetPiece
gameTime.gameTime
gameTime.gameTimeInSec
player.id
player.position
player.positionSide
start.coordinates.x
start.coordinates.y
start.adjCoordinates.x
start.adjCoordinates.y
start.packingZone
start.pitchPosition
start.lane
pass.distance
pass.angle
pass.receiver
pxT.team
pxT.opponent
formation.team
formation.opponent
player
start
pass
pxT
formation
end.coordinates.x
end.coordinates.y
end.adjCoordinates.x
end.adjCoordinates.y
end.packingZone
end.pitchPosition
end.lane
pass.receiver.playerId
pass.receiver.type
duel.duelType
duel.playerId
setPiece.id
setPiece.subPhaseId
setPiece.mainEvent
shot.distance
shot.angle
shot.targetPoint.y
shot.targetPoint.z
shot.targetPoint


## Inspect sample records

Display a small number of events with every column visible.

In [11]:
with pd.option_context("display.max_columns", None):
    display(impect_events.head(15))

    index          id  squadId  pressure         actionType  end  \
0       0  4858179098     33.0       0.0           KICK_OFF  NaN   
1       1  4858179099      NaN       NaN           NO_VIDEO  NaN   
2       2  4858179100     33.0       0.0          RECEPTION  NaN   
3       3  4858179101     33.0       0.0            DRIBBLE  NaN   
4       4  4858179102     33.0       0.0               PASS  NaN   
5       5  4858179103     38.0      90.0  LOOSE_BALL_REGAIN  NaN   
6       6  4858179104     38.0      90.0               PASS  NaN   
7       7  4858179105     38.0      60.0          RECEPTION  NaN   
8       8  4858179106     38.0      60.0          CLEARANCE  NaN   
9       9  4858179107     33.0      70.0  LOOSE_BALL_REGAIN  NaN   
10     10  4858179108     33.0      70.0               PASS  NaN   
11     11  4858179109     33.0       NaN                OUT  NaN   
12     12  4858179110     38.0       0.0           THROW_IN  NaN   
13     13  4858179111     38.0      30.0        

## Inspect coordinates and outcomes

Locate start/end fields and check PASS result labels. Provider SUCCESS, NEUTRAL and FAIL remain distinct.

In [12]:
# Remove the leftover heading from the summary.
midfield_summary.columns.name = None

# Find coordinate fields without printing every event column.
location_columns = [
    column
    for column in impect_events.columns
    if column.startswith(("start.", "end."))
]

print("LOCATION FIELDS")
for column in location_columns:
    print(column)

# Inspect result labels specifically for passes.
passes = impect_events.loc[
    impect_events["actionType"].eq("PASS")
].copy()

print("\nPASS RESULT LABELS")
print(passes["result"].value_counts(dropna=False).to_string())

LOCATION FIELDS
start.coordinates.x
start.coordinates.y
start.adjCoordinates.x
start.adjCoordinates.y
start.packingZone
start.pitchPosition
start.lane
end.coordinates.x
end.coordinates.y
end.adjCoordinates.x
end.adjCoordinates.y
end.packingZone
end.pitchPosition
end.lane

PASS RESULT LABELS
result
SUCCESS    717
FAIL       136
NEUTRAL     64
None         1


## Read the player directory

Load names and IDs independently from the event records.

In [13]:
players_data = metadata(f"players/players_{iteration_id}.json")
print("Player directory records:", len(players_data))


Player directory records: 570


## Attach player names

Join by player ID with a many-to-one validation to avoid multiplying event-summary rows.

In [14]:
players = pd.json_normalize(players_data)

# Build a lookup table: one row per player.
player_lookup = (
    players[["id", "commonname"]]
    .rename(columns={
        "id": "player.id",
        "commonname": "player_name",
    })
)

# Attach names while preserving every midfield-summary row.
midfield_named = (
    midfield_summary.reset_index()
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

print("Rows before join:", len(midfield_summary))
print("Rows after join:", len(midfield_named))
print(
    "Rows without a matching player:",
    midfield_named["_merge"].eq("left_only").sum(),
)

columns_to_display = [
    "squadId",
    "player.id",
    "player_name",
    "player.position",
    "PASS",
    "RECEPTION",
    "DRIBBLE",
    "INTERCEPTION",
    "LOOSE_BALL_REGAIN",
    "GROUND_DUEL",
    "SHOT",
]

display(midfield_named[columns_to_display])

Rows before join: 11
Rows after join: 11
Rows without a matching player: 0
    squadId  player.id  ... GROUND_DUEL SHOT
0      33.0       98.0  ...           0    3
1      33.0      216.0  ...           0    0
2      33.0     1053.0  ...           1    3
3      33.0     7594.0  ...           2    0
4      33.0    53703.0  ...           2    3
5      38.0     1099.0  ...           3    1
6      38.0     3153.0  ...           1    1
7      38.0     4366.0  ...           0    0
8      38.0     6469.0  ...           2    0
9      38.0    13599.0  ...           3    1
10     38.0    23157.0  ...           3    0

[11 rows x 11 columns]


## Inspect team columns

Check the squad directory before building the team lookup.

In [15]:
squads = pd.json_normalize(squads_data)

print("Squad columns:", squads.columns.tolist())
display(squads.head(3))

Squad columns: ['id', 'name', 'countryId', 'type', 'gender', 'imageUrl', 'idMappings', 'access']
   id  ... access
0  27  ...   True
1  29  ...   True
2  30  ...   True

[3 rows x 8 columns]


## Count passing outcomes

Count all pass outcomes, keeping missing results visible instead of silently dropping them.

In [16]:
midfield_passes = midfield_events.loc[
    midfield_events["actionType"].eq("PASS")
].copy()

# Label missing outcomes explicitly so they remain in our counts.
midfield_passes["result_label"] = (
    midfield_passes["result"].fillna("MISSING")
)

pass_summary = (
    midfield_passes.groupby(
        ["squadId", "player.id", "player.position", "result_label"]
    )
    .size()
    .unstack("result_label", fill_value=0)
    .reindex(
        columns=["SUCCESS", "FAIL", "NEUTRAL", "MISSING"],
        fill_value=0,
    )
    .reset_index()
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
)

pass_summary.columns.name = None

pass_summary["total_pass_events"] = pass_summary[
    ["SUCCESS", "FAIL", "NEUTRAL", "MISSING"]
].sum(axis=1)

display(
    pass_summary[
        [
            "player_name",
            "squadId",
            "player.position",
            "SUCCESS",
            "FAIL",
            "NEUTRAL",
            "MISSING",
            "total_pass_events",
        ]
    ]
)

             player_name  squadId  ... MISSING  total_pass_events
0         Joshua Kimmich     33.0  ...       1                 99
1          Thomas Müller     33.0  ...       0                 10
2          Leon Goretzka     33.0  ...       0                 64
3          Konrad Laimer     33.0  ...       0                 11
4          Jamal Musiala     33.0  ...       0                 28
5   Leonardo Bittencourt     38.0  ...       0                 15
6          Romano Schmid     38.0  ...       0                  4
7         Dawid Kownacki     38.0  ...       0                  1
8            Senne Lynen     38.0  ...       0                 15
9             Jens Stage     38.0  ...       0                 20
10        Christian Groß     38.0  ...       0                  5

[11 rows x 8 columns]


## Attach team names

Join the pass summary to the squad lookup and check unmatched IDs.

In [17]:
# One lookup row per team.
squad_lookup = squads[["id", "name"]].rename(
    columns={
        "id": "squadId",
        "name": "team_name",
    }
)

# Attach team names to the pass summary.
pass_summary_named = pass_summary.merge(
    squad_lookup,
    on="squadId",
    how="left",
    validate="many_to_one",
    indicator="team_match",
)

print("Rows before join:", len(pass_summary))
print("Rows after join:", len(pass_summary_named))
print(
    "Rows without a matching team:",
    pass_summary_named["team_match"].eq("left_only").sum(),
)

display(
    pass_summary_named[
        [
            "player_name",
            "team_name",
            "player.position",
            "SUCCESS",
            "FAIL",
            "NEUTRAL",
            "MISSING",
            "total_pass_events",
        ]
    ]
)

Rows before join: 11
Rows after join: 11
Rows without a matching team: 0
             player_name          team_name  ... MISSING  total_pass_events
0         Joshua Kimmich  FC Bayern München  ...       1                 99
1          Thomas Müller  FC Bayern München  ...       0                 10
2          Leon Goretzka  FC Bayern München  ...       0                 64
3          Konrad Laimer  FC Bayern München  ...       0                 11
4          Jamal Musiala  FC Bayern München  ...       0                 28
5   Leonardo Bittencourt   SV Werder Bremen  ...       0                 15
6          Romano Schmid   SV Werder Bremen  ...       0                  4
7         Dawid Kownacki   SV Werder Bremen  ...       0                  1
8            Senne Lynen   SV Werder Bremen  ...       0                 15
9             Jens Stage   SV Werder Bremen  ...       0                 20
10        Christian Groß   SV Werder Bremen  ...       0                  5

[11 rows x 8 c

## Inspect neutral and missing outcomes

Look at example events; these labels are not automatically failed or completed passes.

In [18]:
# Inspect a few neutral passes and the pass with a missing result.
unresolved_passes = passes.loc[
    passes["result"].eq("NEUTRAL")
    | passes["result"].isna()
]

inspection_columns = [
    "index",
    "gameTime.gameTimeInSec",
    "player.id",
    "actionType",
    "result",
    "action",
    "phase",
]

print(
    unresolved_passes[inspection_columns]
    .head(8)
    .to_string(index=False)
)

 index  gameTime.gameTimeInSec  player.id actionType  result            action                phase
     4                  4.1930     1202.0       PASS NEUTRAL     DIAGONAL_PASS        IN_POSSESSION
   113                145.1401     6201.0       PASS NEUTRAL          LOW_PASS        IN_POSSESSION
   146                255.9120     3772.0       PASS NEUTRAL     DIAGONAL_PASS        IN_POSSESSION
   182                325.3511     8451.0       PASS NEUTRAL      CHIPPED_PASS        IN_POSSESSION
   273                571.5910    53703.0       PASS NEUTRAL SHORT_AERIAL_PASS        IN_POSSESSION
   280                579.3360     1028.0       PASS NEUTRAL          LOW_PASS          SECOND_BALL
   330                693.9821    13599.0       PASS NEUTRAL          LOW_PASS        IN_POSSESSION
   446                849.5631    50321.0       PASS NEUTRAL      CHIPPED_PASS ATTACKING_TRANSITION


## Review the named sample

Confirm row counts and readable player/team labels after joining.

In [19]:
print("Rows:", len(pass_summary_named))
print(
    "Unmatched teams:",
    pass_summary_named["team_match"].eq("left_only").sum(),
)

display(
    pass_summary_named[
        ["player_name", "team_name", "total_pass_events"]
    ]
)

Rows: 11
Unmatched teams: 0
             player_name          team_name  total_pass_events
0         Joshua Kimmich  FC Bayern München                 99
1          Thomas Müller  FC Bayern München                 10
2          Leon Goretzka  FC Bayern München                 64
3          Konrad Laimer  FC Bayern München                 11
4          Jamal Musiala  FC Bayern München                 28
5   Leonardo Bittencourt   SV Werder Bremen                 15
6          Romano Schmid   SV Werder Bremen                  4
7         Dawid Kownacki   SV Werder Bremen                  1
8            Senne Lynen   SV Werder Bremen                 15
9             Jens Stage   SV Werder Bremen                 20
10        Christian Groß   SV Werder Bremen                  5


## Inspect player-position aggregates

The provider aggregate export supplies playing time and KPIs at match/team/player/position grain.

In [20]:
player_kpis_data, cache_status = load_or_download_json(
    f"{IMPECT_BASE_URL}/player_kpis/player_kpis_{example_match_id}.json",
    RAW_DIR / "player_kpis" / f"player_kpis_{example_match_id}.json")
print(json.dumps(player_kpis_data, indent=2, ensure_ascii=False)[:5000])


{
  "matchId": 122838,
  "squadHome": {
    "id": 38,
    "players": [
      {
        "id": 1028,
        "position": "RIGHT_WINGBACK_DEFENDER",
        "playDuration": 3695.644,
        "matchShare": 0.62288886,
        "kpis": [
          {
            "kpiId": 0,
            "value": 6.5
          },
          {
            "kpiId": 1,
            "value": 6.0
          },
          {
            "kpiId": 2,
            "value": 1.4
          },
          {
            "kpiId": 3,
            "value": 6.2
          },
          {
            "kpiId": 4,
            "value": 1.4
          },
          {
            "kpiId": 7,
            "value": 15.1
          },
          {
            "kpiId": 8,
            "value": 8.0
          },
          {
            "kpiId": 9,
            "value": 2.6
          },
          {
            "kpiId": 10,
            "value": 2.01
          },
          {
            "kpiId": 15,
            "value": 6.0
          },
          {
            

## Extract playing time

Build duration rows at exactly that grain. A player can have several position rows in one match.

In [21]:
duration_rows = []

for squad_key in ["squadHome", "squadAway"]:
    squad = player_kpis_data[squad_key]

    for player_record in squad["players"]:
        duration_rows.append({
            "match_id": player_kpis_data["matchId"],
            "squadId": squad["id"],
            "player.id": player_record["id"],
            "player.position": player_record["position"],
            "playDuration": player_record["playDuration"],
            "matchShare": player_record["matchShare"],
        })

player_durations = pd.DataFrame(duration_rows)

player_durations = (
    player_durations
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        squad_lookup,
        on="squadId",
        how="left",
        validate="many_to_one",
    )
)

display(
    player_durations[
        [
            "player_name",
            "team_name",
            "player.position",
            "playDuration",
            "matchShare",
        ]
    ].sort_values(["team_name", "player_name"])
)

print(
    "Duplicate match/team/player/position records:",
    player_durations.duplicated(
        ["match_id", "squadId", "player.id", "player.position"]
    ).sum(),
)

                 player_name          team_name  ... playDuration  matchShare
28           Alphonso Davies  FC Bayern München  ...     5933.071    1.000000
26           Dayot Upamecano  FC Bayern München  ...     5933.071    1.000000
21  Eric-Maxim Choupo-Moting  FC Bayern München  ...      750.629    0.126516
19                Harry Kane  FC Bayern München  ...     5182.442    0.873484
30             Jamal Musiala  FC Bayern München  ...     5182.442    0.873484
16            Joshua Kimmich  FC Bayern München  ...     5933.071    1.000000
20            Kingsley Coman  FC Bayern München  ...     5182.442    0.873484
24             Konrad Laimer  FC Bayern München  ...     1027.412    0.173167
22             Leon Goretzka  FC Bayern München  ...     4905.659    0.826833
17                Leroy Sané  FC Bayern München  ...     5933.071    1.000000
31                Mathys Tel  FC Bayern München  ...      750.629    0.126516
25          Matthijs de Ligt  FC Bayern München  ...     1732.28

## Calculate a sample rate per 90

Divide actions by playing time at the same recorded position. Zero or missing minutes cannot support a rate.

In [22]:
# Convert seconds to minutes without changing the original duration.
player_durations["minutes_played"] = (
    player_durations["playDuration"] / 60
)

# Match action counts to time played at the SAME position.
midfield_per90 = midfield_summary.reset_index().merge(
    player_durations[
        [
            "squadId",
            "player.id",
            "player.position",
            "player_name",
            "team_name",
            "minutes_played",
        ]
    ],
    on=["squadId", "player.id", "player.position"],
    how="left",
    validate="one_to_one",
    indicator="duration_match",
)

print(
    "Rows without matching duration:",
    midfield_per90["duration_match"].eq("left_only").sum(),
)

# Missing or zero minutes cannot support a per-90 calculation.
valid_minutes = midfield_per90["minutes_played"].where(
    midfield_per90["minutes_played"].gt(0)
)

midfield_per90["pass_events_per90"] = (
    midfield_per90["PASS"] / valid_minutes * 90
)

display(
    midfield_per90[
        [
            "player_name",
            "team_name",
            "player.position",
            "minutes_played",
            "PASS",
            "pass_events_per90",
        ]
    ].round(2)
)

Rows without matching duration: 0
             player_name          team_name  ... PASS  pass_events_per90
0         Joshua Kimmich  FC Bayern München  ...   99              90.11
1          Thomas Müller  FC Bayern München  ...   10              71.94
2          Leon Goretzka  FC Bayern München  ...   64              70.45
3          Konrad Laimer  FC Bayern München  ...   11              57.82
4          Jamal Musiala  FC Bayern München  ...   28              29.18
5   Leonardo Bittencourt   SV Werder Bremen  ...   15              19.15
6          Romano Schmid   SV Werder Bremen  ...    4              12.68
7         Dawid Kownacki   SV Werder Bremen  ...    1               5.12
8            Senne Lynen   SV Werder Bremen  ...   15              19.15
9             Jens Stage   SV Werder Bremen  ...   20              22.14
10        Christian Groß   SV Werder Bremen  ...    5              15.84

[11 rows x 6 columns]
